# Attention Pattern Interpretation

Detect and classify attention motifs: previous-token, self-attention, BOS,
uniform patterns. Profile head functions and track pattern evolution.

## Why This Matters

Attention pattern interpretation reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_pattern_interpretation import (
    detect_attention_motifs, attention_pattern_summary,
    head_function_profile, all_heads_motif_classification,
    attention_pattern_evolution,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Detect Attention Motifs

What kind of pattern does this head use?

In [ ]:
result = detect_attention_motifs(model, tokens, layer=0, head=0)
print(f"Head L{result['layer']}H{result['head']}")
print(f"Dominant motif: {result['dominant_motif']}\n")
for motif, score in sorted(result['motif_scores'].items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 30)
    print(f"  {motif:18s}: {score:.3f} {bar}")

## Attention Pattern Summary

Quick statistics about a head's attention pattern.

In [ ]:
result = attention_pattern_summary(model, tokens, layer=0, head=0)
sparse_str = 'SPARSE' if result['is_sparse'] else 'dense'
print(f"Entropy: {result['mean_entropy']:.4f}")
print(f"Sparsity: {result['sparsity']:.4f} [{sparse_str}]")
print(f"Mean distance: {result['mean_distance']:.4f}")
print(f"Mean max attention: {result['mean_max_attention']:.4f}")

## Head Function Profile

What does this head contribute to the output?

In [ ]:
result = head_function_profile(model, tokens, layer=0, head=0)
cons_str = 'CONSISTENT' if result['is_consistent'] else 'variable'
print(f"Output norm: {result['mean_output_norm']:.4f}")
print(f"Logit magnitude: {result['mean_logit_magnitude']:.4f}")
print(f"Direction consistency: {result['direction_consistency']:.4f} [{cons_str}]")

## All-Heads Motif Classification

Classify every head in the model.

In [ ]:
result = all_heads_motif_classification(model, tokens)
print(f"Total heads: {result['n_heads_total']}")
print(f"Motif distribution: {result['motif_counts']}\n")
for h in result['per_head']:
    print(f"  L{h['layer']}H{h['head']}: {h['dominant_motif']:18s} "
          f"(conf={h['confidence']:.3f})")

## Attention Pattern Evolution

How does head 0's pattern change across layers?

In [ ]:
result = attention_pattern_evolution(model, tokens, head=0)
stable_str = 'STABLE' if result['is_stable'] else 'variable'
print(f"Mean stability: {result['mean_stability']:.4f} [{stable_str}]\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: entropy={p['mean_entropy']:.4f}, "
          f"max_attn={p['mean_max_attention']:.4f}, "
          f"sim_prev={p['similarity_to_previous']:.4f}")